# YAGO 3-10+

In [2]:
from glob import glob
from tqdm import tqdm
from pickle import dump, load, HIGHEST_PROTOCOL
from os import makedirs
from os.path import exists
from urllib.request import urlopen
from bz2 import open as bz2open
from shutil import copy as copy_file
import gzip
import tarfile
from subprocess import run, PIPE

from random import randint

!pip install py7zr

In [3]:
draft_folder=f"./draft/YAGO3-10+"
destination_folder=f"./YAGO3-10+"

In [4]:
draft_folder=f"{draft_folder}{'' if draft_folder.endswith('/') else '/'}"
destination_folder=f"{destination_folder}{'' if destination_folder.endswith('/') else '/'}"

YAGO3_10_LP_DATASET_FOLDER="https://github.com/TimDettmers/ConvE/raw/refs/heads/master/"
YAGO3_10_LP_DATASET_FILE="YAGO3-10.tar.gz"
YAGO3_10_ARCHIVE_URL_FOLDER = "https://yago-knowledge.org/data/yago3/"
YAGO3_10_ARCHIVE_URL_FILE="yago-3.0.2-turtle-simple.7z"

YAGO3_10_LP=f"{draft_folder}LP/"
YAGO3_10=f"{draft_folder}YAGO3-10/"
YAGO3=f"{draft_folder}YAGO3/"

dest_txt_folder=destination_folder
dest_pkl_folder=f"{destination_folder}pickle/"

In [5]:
for folder in [
    draft_folder,
    dest_txt_folder,
    dest_pkl_folder,
    YAGO3_10_LP,
    YAGO3_10,
    YAGO3
]: 
    makedirs(folder, exist_ok=True)

In [6]:
if not exists(f"{draft_folder}{YAGO3_10_LP_DATASET_FILE[:-7]}/train.txt") \
    or not exists(f"{draft_folder}{YAGO3_10_LP_DATASET_FILE[:-7]}/test.txt") \
    or not exists(f"{draft_folder}{YAGO3_10_LP_DATASET_FILE[:-7]}/train.txt"):

    if not exists(f"{YAGO3_10_LP_DATASET_FOLDER}{YAGO3_10_LP_DATASET_FILE}"):

        with urlopen(f"{YAGO3_10_LP_DATASET_FOLDER}{YAGO3_10_LP_DATASET_FILE}") as file_online:
            with open(f"{draft_folder}{YAGO3_10_LP_DATASET_FILE}", "wb") as file_local:
                file_local.write(file_online.read())
    
    tar = tarfile.open(f"{draft_folder}{YAGO3_10_LP_DATASET_FILE}", 'r:gz')
    tar.extractall(f"{draft_folder}{YAGO3_10_LP_DATASET_FILE[:-7]}")
    tar.close()

In [7]:
if not exists(f"{YAGO3}{YAGO3_10_ARCHIVE_URL_FILE}"):
    with urlopen(f"{YAGO3_10_ARCHIVE_URL_FOLDER}{YAGO3_10_ARCHIVE_URL_FILE}") as file_online:
        with open(f"{YAGO3}{YAGO3_10_ARCHIVE_URL_FILE}", "wb") as file_local:
            file_local.write(file_online.read())

In [8]:
if not exists(f"{YAGO3}yago-3.0.2-turtle-simple/yagoTypes.ttl") \
    or not exists(f"{YAGO3}yago-3.0.2-turtle-simple/yagoSchema.ttl") \
    or not exists(f"{YAGO3}yago-3.0.2-turtle-simple/yagoTaxonomy.ttl"):
    makedirs(f"{YAGO3}{YAGO3_10_ARCHIVE_URL_FILE}"[:-3], exist_ok=True)
    
    packed_file=f"{YAGO3}{YAGO3_10_ARCHIVE_URL_FILE}"
    unpack_folder=packed_file[:-3]
    
    print(f"py7zr x {packed_file} {unpack_folder}")
    unpack_command = run(["py7zr", "x", packed_file, unpack_folder], stdout = PIPE, stderr = PIPE)
    
    print("Standard output:")
    print(unpack_command.stdout)
    
    if "stderr" in dir(unpack_command):
        print("Standard error:")
        print(unpack_command.stderr)

# ent2id, rel2id, class2id

In [10]:
rels = set([])
ents = set([])

for file in [
    "train.txt",
    "test.txt",
    "valid.txt"
]:
    file_path=f"{YAGO3_10}{file}"
    nb_lines=sum(1 for _ in open(file_path, "r", encoding="utf-8"))
    with open(file_path, "r", encoding="utf-8") as split:
        with tqdm(enumerate(split), total=nb_lines) as bar:
            bar.set_description(f"Extracting entities and relations from {file}")
            for i, line in bar:
                s, p, o = line.strip().split("\t")
    
                s=f"http://yago-knowledge.org/resource/{s}"
                p=f"http://yago-knowledge.org/resource/{p}"
                o=f"http://yago-knowledge.org/resource/{o}"
    
                ents.add(s)
                ents.add(o)
                rels.add(p)

Extracting entities and relations from valid.txt: 100%|████████████████████████| 5000/5000 [00:00<00:00, 319917.01it/s]


In [11]:
len(list(ents)), len(list(rels))

(123182, 37)

In [12]:
rel2id = {rel: i for i, rel in enumerate(list(rels))}
id2rel = {i: rel for rel, i in rel2id.items()}

with open(f"{dest_pkl_folder}rel2id.pkl", "wb") as handle:
    dump(rel2id, handle)

ent2id = {ent: i for i, ent in enumerate(list(ents))}
id2ent = {i: ent for ent, i in ent2id.items()}

with open(f"{dest_pkl_folder}ent2id.pkl", "wb") as handle:
    dump(ent2id, handle)

In [13]:
{k: v for k, v in list(ent2id.items())[:10]}

{'http://yago-knowledge.org/resource/Şanver_Göymen': 0,
 'http://yago-knowledge.org/resource/Qatar_national_under-23_football_team': 1,
 'http://yago-knowledge.org/resource/Main-Spessart': 2,
 'http://yago-knowledge.org/resource/Mariano_Ferreira_Filho': 3,
 'http://yago-knowledge.org/resource/Billy_Squier': 4,
 'http://yago-knowledge.org/resource/Armageddon_(1998_film)': 5,
 'http://yago-knowledge.org/resource/Åge_Hareide': 6,
 'http://yago-knowledge.org/resource/Spyros_Vallas': 7,
 'http://yago-knowledge.org/resource/Nicola_Campedelli': 8,
 'http://yago-knowledge.org/resource/Egri_FC': 9}

In [14]:
{k: v for k, v in list(rel2id.items())[:10]}

{'http://yago-knowledge.org/resource/wasBornIn': 0,
 'http://yago-knowledge.org/resource/livesIn': 1,
 'http://yago-knowledge.org/resource/playsFor': 2,
 'http://yago-knowledge.org/resource/hasMusicalRole': 3,
 'http://yago-knowledge.org/resource/graduatedFrom': 4,
 'http://yago-knowledge.org/resource/isKnownFor': 5,
 'http://yago-knowledge.org/resource/directed': 6,
 'http://yago-knowledge.org/resource/actedIn': 7,
 'http://yago-knowledge.org/resource/isConnectedTo': 8,
 'http://yago-knowledge.org/resource/diedIn': 9}

### Encode train, test, valid splits

In [16]:
inverse_predicate_offset = max(rel2id.values()) + 1
for split in ["train", "test", "valid"]:
    nb_lines=sum(1 for _ in open(f"{YAGO3_10}{split}.txt", "r", encoding="utf-8"))
    with open(f"{YAGO3_10}{split}.txt", "r", encoding='utf-8') as r:
        with open(f"{dest_txt_folder}{split}2id.txt", "w+", encoding='utf-8') as w:
            with tqdm(enumerate(r), total=nb_lines) as bar:
                bar.set_description(f"Encoding {split}2id.txt")
                for i, line in bar:
                    s, p, o = line.strip().split("\t")

                    s=f"http://yago-knowledge.org/resource/{s}"
                    p=f"http://yago-knowledge.org/resource/{p}"
                    o=f"http://yago-knowledge.org/resource/{o}"
                    
                    encoded_s = ent2id[s]
                    encoded_p = rel2id[p]
                    encoded_o = ent2id[o]
                    
                    w.write(f"{encoded_s}\t{encoded_p}\t{encoded_o}\n")   
                    
        copy_file(f"{dest_txt_folder}{split}2id.txt", f"{dest_txt_folder}{split}2id_inv.txt")
        with open(f"{dest_txt_folder}{split}2id.txt", "r", encoding='utf-8') as r:
            with open(f"{dest_txt_folder}{split}2id_inv.txt", "a", encoding='utf-8') as a:
                with tqdm(enumerate(r), total=nb_lines) as bar:
                    bar.set_description(f"Encoding {split}2id_inv.txt")
                    for i, line in bar:
                        s, p, o = line.strip().split("\t")
                        p = int(p)
                        a.write(f"{o}\t{p+inverse_predicate_offset}\t{s}\n")

Encoding valid2id_inv.txt: 100%|███████████████████████████████████████████████| 5000/5000 [00:00<00:00, 319536.80it/s]


# observed_heads_original_kg, observed_tails_original_kg, observed_heads_inv, observed_tails_inv

In [18]:
observed_heads_original_kg={}
observed_heads_inv={}
observed_tails_original_kg={}
observed_tails_inv={}

def observe(d, a, b, c):
    if not a in d.keys():
        d[a]={}

    if not b in d[a].keys():
        d[a][b]=[c]
    else:
        d[a][b].append(c)


for split in ["train", "test", "valid"]:
    nb_lines=sum(1 for _ in open(f"{dest_txt_folder}{split}2id.txt", "r", encoding="utf-8"))
    with open(f"{dest_txt_folder}{split}2id.txt", "r", encoding='utf-8') as r:
        with tqdm(enumerate(r), total=nb_lines) as bar:
            bar.set_description(f"Indexing triples from {split}2id.txt")
            for i, line in bar:
                s, p, o = line.strip().split("\t")
                s, p, o = int(s), int(p), int(o)

                observe(observed_tails_original_kg, s, p, o)
                observe(observed_tails_inv, s, p, o)
                observe(observed_tails_inv, o, p + inverse_predicate_offset, s)

                observe(observed_heads_original_kg, o, p, s)
                observe(observed_heads_inv, o, p, s)
                observe(observed_heads_inv, s, p + inverse_predicate_offset, o)

Indexing triples from valid2id.txt: 100%|██████████████████████████████████████| 5000/5000 [00:00<00:00, 105645.18it/s]


In [19]:
with open(f"{dest_pkl_folder}observed_heads_original_kg.pkl", "wb") as handle:
    dump(observed_heads_original_kg, handle)

with open(f"{dest_pkl_folder}observed_heads_inv.pkl", "wb") as handle:
    dump(observed_heads_inv, handle)

with open(f"{dest_pkl_folder}observed_tails_original_kg.pkl", "wb") as handle:
    dump(observed_tails_original_kg, handle)

with open(f"{dest_pkl_folder}observed_tails_inv.pkl", "wb") as handle:
    dump(observed_tails_inv, handle)

# classid

In [21]:
instances={}

nb_lines=sum(1 for _ in open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoTypes.ttl", "r", encoding="utf-8"))
with open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoTypes.ttl", "r", encoding='utf-8') as r:
    with tqdm(enumerate(r), total=nb_lines) as bar:
        for i, line in bar:
            if not line.startswith("<"):
                continue
            s, p, o, *_ = line.strip().split("\t")

            if not p == "rdf:type":
                continue

            s = f"http://yago-knowledge.org/resource/{s[1:-1]}"
            o = f"http://yago-knowledge.org/resource/{o[1:-1]}"

            if not s in ents:
                continue

            if not s in instances.keys():
                instances[s]=set([o])
            else:
                instances[s].add(o)

100%|██████████████████████████████████████████████████████████████████| 33854055/33854055 [00:59<00:00, 567447.32it/s]


In [22]:
{k: list(v)[:5] for k, v in list(instances.items())[:5]}

{'http://yago-knowledge.org/resource/JS_Bordj_Ménaïel': ['http://yago-knowledge.org/resource/wikicat_Association_football_clubs_established_in_1932',
  'http://yago-knowledge.org/resource/wordnet_club_108227214',
  'http://yago-knowledge.org/resource/wikicat_Sports_clubs_in_Algeria',
  'http://yago-knowledge.org/resource/wikicat_Football_clubs_in_Algeria'],
 'http://yago-knowledge.org/resource/Eddie_Kaye_Thomas': ['http://yago-knowledge.org/resource/wikicat_Jewish_actors',
  'http://yago-knowledge.org/resource/wikicat_American_male_stage_actors',
  'http://yago-knowledge.org/resource/wikicat_American_actors',
  'http://yago-knowledge.org/resource/wikicat_Jewish_American_male_actors',
  'http://yago-knowledge.org/resource/wordnet_person_100007846'],
 'http://yago-knowledge.org/resource/Ahmet_Brković': ['http://yago-knowledge.org/resource/wikicat_Croatian_footballers',
  'http://yago-knowledge.org/resource/wikicat_Leyton_Orient_F.C._players',
  'http://yago-knowledge.org/resource/wikicat

In [23]:
print("Nb of entity in total:", len(ents))
print("Nb of entity with at least a known type:", len(instances.items()))
print("Nb of entity without any known type:", len(ents) - len(instances.items()))

Nb of entity in total: 123182
Nb of entity with at least a known type: 121009
Nb of entity without any known type: 2173


In [24]:
nb_types=sorted(list(set([len(v) for v in instances.values()])))
print("Nb of types per entity", min(nb_types), max(nb_types))

Nb of types per entity 1 101


# Domain / Range information

In [26]:
subproperties={}

nb_lines=sum(1 for _ in open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoSchema.ttl", "r", encoding="utf-8"))
with open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoSchema.ttl", "r", encoding='utf-8') as r:
    with tqdm(enumerate(r), total=nb_lines) as bar:
        for i, line in bar:
            if not line.startswith("<"):
                continue
            s, p, o, *_ = line.strip().split("\t")

            if not p == "rdfs:subPropertyOf":
                continue

            if not s.startswith("<"):
                continue

            if not o.startswith("<"):
                continue

            s = f"http://yago-knowledge.org/resource/{s[1:-1]}"
            o = f"http://yago-knowledge.org/resource/{o[1:-1]}"

            if p == "rdfs:subPropertyOf":
                if not s in subproperties.keys():
                    subproperties[s]=set([])
                subproperties[s].add(o)

100%|████████████████████████████████████████████████████████████████████████████████████████| 985/985 [00:00<?, ?it/s]


In [27]:
def subsumption_closure(c, hierarchy):
    if not c in hierarchy.keys():
        return [c]
    else:
        result = [c]
        for sc in hierarchy[c]:
            result.extend(subsumption_closure(sc, hierarchy))
        return result

In [28]:
subproperty_closure={
    rel: set(subsumption_closure(rel, subproperties))
    for rel in rels
}

In [29]:
properties_of_interest=set(subproperty_closure.keys()).union(*list(subproperty_closure.values()))

In [30]:
domains={}
ranges={}

nb_lines=sum(1 for _ in open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoSchema.ttl", "r", encoding="utf-8"))
with open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoSchema.ttl", "r", encoding='utf-8') as r:
    with tqdm(enumerate(r), total=nb_lines) as bar:
        for i, line in bar:
            if not line.startswith("<"):
                continue
            s, p, o, *_ = line.strip().split("\t")

            if not p in ["rdfs:domain", "rdfs:range", "rdfs:subPropertyOf"]:
                continue

            if not s.startswith("<"):
                continue

            if not o.startswith("<"):
                continue

            s = f"http://yago-knowledge.org/resource/{s[1:-1]}"
            o = f"http://yago-knowledge.org/resource/{o[1:-1]}"

            if not s in properties_of_interest:
                continue

            if p == "rdfs:domain":
                if not s in domains.keys():
                    domains[s]=set([])
                domains[s].add(o)

            if p == "rdfs:range":
                if not s in ranges.keys():
                    ranges[s]=set([])
                ranges[s].add(o)

100%|████████████████████████████████████████████████████████████████████████████████████████| 985/985 [00:00<?, ?it/s]


In [31]:
domains={
    rel: set([]).union(*[
        domains.get(super_rel, set([]))
        for super_rel in subproperty_closure[rel]
    ])
    for rel in rels
}
domains={k: v for k, v in domains.items() if len(v) > 0}

ranges={
    rel: set([]).union(*[
        ranges.get(super_rel, set([]))
        for super_rel in subproperty_closure[rel]
    ])
    for rel in rels
}
ranges={k: v for k, v in ranges.items() if len(v) > 0}

In [32]:
print("Max number of domains per property:", max([len(x) for x in domains.values()]))
print("Max number of ranges per property:", max([len(x) for x in ranges.values()]))

Max number of domains per property: 1
Max number of ranges per property: 1


In [33]:
print("Nb of predicates", len(rels))
print("Nb of domained predicates", len(domains.keys()))
print("Nb of ranged predicates", len(ranges.keys()))

Nb of predicates 37
Nb of domained predicates 37
Nb of ranged predicates 28


In [34]:
{k: v for k, v in list(domains.items())[:5]}

{'http://yago-knowledge.org/resource/wasBornIn': {'http://yago-knowledge.org/resource/wordnet_person_100007846'},
 'http://yago-knowledge.org/resource/livesIn': {'http://yago-knowledge.org/resource/wordnet_person_100007846'},
 'http://yago-knowledge.org/resource/playsFor': {'http://yago-knowledge.org/resource/wordnet_person_100007846'},
 'http://yago-knowledge.org/resource/hasMusicalRole': {'http://yago-knowledge.org/resource/wordnet_person_100007846'},
 'http://yago-knowledge.org/resource/graduatedFrom': {'http://yago-knowledge.org/resource/wordnet_person_100007846'}}

In [35]:
{k: v for k, v in list(ranges.items())[:5]}

{'http://yago-knowledge.org/resource/wasBornIn': {'http://yago-knowledge.org/resource/wordnet_city_108524735'},
 'http://yago-knowledge.org/resource/livesIn': {'http://yago-knowledge.org/resource/wordnet_location_100027167'},
 'http://yago-knowledge.org/resource/playsFor': {'http://yago-knowledge.org/resource/wordnet_organization_108008335'},
 'http://yago-knowledge.org/resource/graduatedFrom': {'http://yago-knowledge.org/resource/wordnet_university_108286569'},
 'http://yago-knowledge.org/resource/directed': {'http://yago-knowledge.org/resource/wordnet_movie_106613686'}}

In [36]:
domain_inferences={}

for s, po in observed_tails_original_kg.items():
    domain_inferences[id2ent[s]]=set([])
    for p, _ in po.items():
        domain_inferences[id2ent[s]] = domain_inferences[id2ent[s]].union(domains.get(id2rel[p], set([])))

domain_inferences={k: v for k, v in domain_inferences.items() if len(v) > 0}

In [37]:
n=randint(0, len(domain_inferences.keys())-6)
{k: v for k, v in list(domain_inferences.items())[n:n+5]}

{'http://yago-knowledge.org/resource/Candy_(1968_film)': {'http://yago-knowledge.org/resource/yagoPermanentlyLocatedEntity'},
 'http://yago-knowledge.org/resource/Kingdom_of_Poland_(1385–1569)': {'http://yago-knowledge.org/resource/wordnet_location_100027167',
  'http://yago-knowledge.org/resource/yagoLegalActorGeo',
  'http://yago-knowledge.org/resource/yagoPermanentlyLocatedEntity'},
 'http://yago-knowledge.org/resource/Antônio_Pinto_(composer)': {'http://yago-knowledge.org/resource/wordnet_person_100007846'},
 'http://yago-knowledge.org/resource/Dick_Haymes': {'http://yago-knowledge.org/resource/wordnet_person_100007846'},
 'http://yago-knowledge.org/resource/Ullensaker': {'http://yago-knowledge.org/resource/yagoPermanentlyLocatedEntity'}}

In [38]:
range_inferences={}

for o, ps in observed_heads_original_kg.items():
    range_inferences[id2ent[o]]=set([])
    for p, _ in ps.items():
        range_inferences[id2ent[o]] = range_inferences[id2ent[o]].union(ranges.get(id2rel[p], set([])))

range_inferences={k: v for k, v in range_inferences.items() if len(v) > 0}

In [39]:
n=randint(0, len(range_inferences.keys())-6)
{k: v for k, v in list(range_inferences.items())[n:n+5]}

{'http://yago-knowledge.org/resource/FC_Veris': {'http://yago-knowledge.org/resource/wordnet_organization_108008335',
  'http://yago-knowledge.org/resource/yagoLegalActor'},
 'http://yago-knowledge.org/resource/Bedford_Town_F.C.': {'http://yago-knowledge.org/resource/wordnet_organization_108008335',
  'http://yago-knowledge.org/resource/yagoLegalActor'},
 'http://yago-knowledge.org/resource/Moncton_Wildcats': {'http://yago-knowledge.org/resource/yagoLegalActor'},
 'http://yago-knowledge.org/resource/Galatasaray_S.K.': {'http://yago-knowledge.org/resource/wordnet_organization_108008335',
  'http://yago-knowledge.org/resource/yagoLegalActor'},
 'http://yago-knowledge.org/resource/New_York_Express': {'http://yago-knowledge.org/resource/wordnet_organization_108008335',
  'http://yago-knowledge.org/resource/yagoLegalActor'}}

In [40]:
classes_of_interest=set([]).union(*instances.values()).union(*domain_inferences.values()).union(*range_inferences.values())

In [41]:
len(classes_of_interest)

91763

In [42]:
subclasses={}

nb_lines=sum(1 for _ in open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoTaxonomy.ttl", "r", encoding="utf-8"))
with open(f"{YAGO3}yago-3.0.2-turtle-simple/yagoTaxonomy.ttl", "r", encoding='utf-8') as r:
    with tqdm(enumerate(r), total=nb_lines) as bar:
        for i, line in bar:
            if not line.startswith("<"):
                continue
            s, p, o, *_ = line.strip().split("\t")

            if not p == "rdfs:subClassOf":
                continue

            if not s.startswith("<"):
                continue

            if not o.startswith("<"):
                continue

            s = f"http://yago-knowledge.org/resource/{s[1:-1]}"
            o = f"http://yago-knowledge.org/resource/{o[1:-1]}"

            if not s in subclasses.keys():
                subclasses[s]=set([])

            subclasses[s].add(o)

100%|████████████████████████████████████████████████████████████████████| 1140933/1140933 [00:04<00:00, 232869.62it/s]


In [43]:
{k: v for k, v in list(subclasses.items())[:5]}

{'http://yago-knowledge.org/resource/wordnet_agape_101028534': {'http://yago-knowledge.org/resource/wordnet_religious_ceremony_101028082'},
 'http://yago-knowledge.org/resource/wikicat_Peace_treaties_of_Great_Britain': {'http://yago-knowledge.org/resource/wordnet_peace_106773976'},
 'http://yago-knowledge.org/resource/wikicat_Former_National_Forests_of_the_United_States': {'http://yago-knowledge.org/resource/wordnet_forest_108438533'},
 'http://yago-knowledge.org/resource/wikicat_Defunct_schools_in_Kingston_upon_Thames': {'http://yago-knowledge.org/resource/wordnet_school_108276720'},
 'http://yago-knowledge.org/resource/wikicat_Islands_of_Tierra_del_Fuego': {'http://yago-knowledge.org/resource/wordnet_island_109316454'}}

In [44]:
subclass_closure={
    cls: set(subsumption_closure(cls, subclasses))
    for cls in classes_of_interest
}

In [45]:
len(subclass_closure.keys())

91763

In [46]:
{k: list(v)[:3] for k, v in list(subclass_closure.items())[:5]}

{'http://yago-knowledge.org/resource/wikicat_Austrian_socialists': ['http://yago-knowledge.org/resource/wordnet_object_100002684',
  'http://yago-knowledge.org/resource/wordnet_living_thing_100004258',
  'http://yago-knowledge.org/resource/wordnet_whole_100003553'],
 'http://yago-knowledge.org/resource/wikicat_People_from_Haltern': ['http://yago-knowledge.org/resource/wordnet_object_100002684',
  'http://yago-knowledge.org/resource/wordnet_living_thing_100004258',
  'http://yago-knowledge.org/resource/wordnet_whole_100003553'],
 'http://yago-knowledge.org/resource/wikicat_County_seats_in_Wisconsin': ['http://yago-knowledge.org/resource/wordnet_center_108523483',
  'http://yago-knowledge.org/resource/wordnet_object_100002684',
  'http://yago-knowledge.org/resource/wordnet_county_seat_108547143'],
 'http://yago-knowledge.org/resource/wikicat_German_botanists': ['http://yago-knowledge.org/resource/wordnet_object_100002684',
  'http://yago-knowledge.org/resource/wordnet_living_thing_100004

In [47]:
classes_set=set(subclass_closure.keys()).union(*subclass_closure.values())

abstracts_classes=[c for c in classes_set if not c in subclass_closure.keys()]
abstracts_classes_closure={
    cls: set(subsumption_closure(cls, subclasses))
    for cls in abstracts_classes
}

subclass_closure.update(abstracts_classes_closure)

In [48]:
subclass_closure["http://yago-knowledge.org/resource/wordnet_soap_opera_106622020"]

{'http://yago-knowledge.org/resource/wordnet_abstraction_100002137',
 'http://yago-knowledge.org/resource/wordnet_broadcast_106619428',
 'http://yago-knowledge.org/resource/wordnet_event_100029378',
 'http://yago-knowledge.org/resource/wordnet_psychological_feature_100023100',
 'http://yago-knowledge.org/resource/wordnet_serial_106621447',
 'http://yago-knowledge.org/resource/wordnet_show_106619065',
 'http://yago-knowledge.org/resource/wordnet_soap_opera_106622020',
 'http://yago-knowledge.org/resource/wordnet_social_event_107288639',
 'http://yago-knowledge.org/resource/yagoPermanentlyLocatedEntity'}

In [49]:
subclassof={k: v for k, v in subclasses.items() if k in classes_set}

In [50]:
len(list(subclassof.items()))

93932

In [51]:
subclassof={k: set([item for item in v if item in classes_set]) for k, v in subclassof.items()}

In [52]:
len(list(subclassof.items()))

93932

In [53]:
subclassof={k: v for k, v in subclassof.items() if len(list(v)) > 0}

In [54]:
len(list(subclassof.items()))

93932

In [55]:
{k: v for k, v in list(subclassof.items())[:5]}

{'http://yago-knowledge.org/resource/wikicat_British_television_soap_operas': {'http://yago-knowledge.org/resource/wordnet_soap_opera_106622020'},
 'http://yago-knowledge.org/resource/wordnet_hearer_110165448': {'http://yago-knowledge.org/resource/wordnet_perceiver_109626589'},
 'http://yago-knowledge.org/resource/wikicat_Companies_operating_former_Canadian_Pacific_Railway_lines%26lt%3B%21--ex-Soo_south_of_Superior--%26gt%3B': {'http://yago-knowledge.org/resource/wordnet_company_108058098'},
 'http://yago-knowledge.org/resource/wikicat_Writers_from_Hunan': {'http://yago-knowledge.org/resource/wordnet_writer_110794014'},
 'http://yago-knowledge.org/resource/wikicat_Populated_places_in_McHenry_County,_Illinois': {'http://yago-knowledge.org/resource/wordnet_site_108651247'}}

In [56]:
class2id={cls: i for i, cls in enumerate(classes_set)}
id2class = {i: cls for cls, i in class2id.items()}

with open(f"{dest_pkl_folder}class2id.pkl", "wb") as handle:
    dump(class2id, handle)

In [57]:
subclassof2id={class2id[k]: set([class2id[item] for item in v]) for k, v in subclassof.items() if len(list(v)) > 0}

with open(f"{dest_pkl_folder}subclassof2id.pkl", "wb") as handle:
    dump(subclassof2id, handle)

In [58]:
rid2domid = {rel2id[rel]: class2id[list(classes)[0]] for rel, classes in domains.items()}
{k: v for k, v in list(rid2domid.items())[:5]}

{0: 32846, 1: 32846, 2: 32846, 3: 32846, 4: 32846}

In [59]:
rid2rangeid = {rel2id[rel]: class2id[list(classes)[0]] for rel, classes in ranges.items()}
{k: v for k, v in list(rid2rangeid.items())[:5]}

{0: 3234, 1: 69920, 2: 43746, 4: 33476, 6: 81673}

In [60]:
with open(f"{dest_pkl_folder}rid2domid.pkl", "wb") as handle:
    dump({k: set([v]) for k, v in rid2domid.items()}, handle)

with open(f"{dest_pkl_folder}rid2rangeid.pkl", "wb") as handle:
    dump({k: set([v]) for k, v in rid2rangeid.items()}, handle)

In [61]:
instance_no_closure={
    ent: [
        cls
        for cls in instances.get(ent, set([]))\
                    .union(domain_inferences.get(ent, set([])))\
                    .union(range_inferences.get(ent, set([])))
    ]
    for ent in tqdm(ents)
}

100%|██████████████████████████████████████████████████████████████████████| 123182/123182 [00:00<00:00, 129390.67it/s]


In [62]:
{k: list(v)[:5] for k, v in list(instance_no_closure.items())[:5]}

{'http://yago-knowledge.org/resource/Şanver_Göymen': ['http://yago-knowledge.org/resource/wikicat_Turkish_football_managers',
  'http://yago-knowledge.org/resource/wikicat_Turkey_international_footballers',
  'http://yago-knowledge.org/resource/wikicat_Association_football_goalkeepers',
  'http://yago-knowledge.org/resource/wikicat_Turkish_people',
  'http://yago-knowledge.org/resource/wikicat_Dardanelspor_footballers'],
 'http://yago-knowledge.org/resource/Qatar_national_under-23_football_team': ['http://yago-knowledge.org/resource/wikicat_National_sports_teams_of_Qatar',
  'http://yago-knowledge.org/resource/yagoLegalActor',
  'http://yago-knowledge.org/resource/wikicat_Asian_national_under-23_association_football_teams',
  'http://yago-knowledge.org/resource/wordnet_organization_108008335',
  'http://yago-knowledge.org/resource/wordnet_football_team_108080025'],
 'http://yago-knowledge.org/resource/Main-Spessart': ['http://yago-knowledge.org/resource/yagoGeoEntity',
  'http://yago-k

In [63]:
inst_type={
    ent2id[ent]: set([class2id[cls] for cls in classes])
    for ent, classes in instance_no_closure.items()
}

In [64]:
{k: list(v)[:5] for k, v in list(inst_type.items())[:5]}

{0: [54338, 71683, 62822, 63974, 76326],
 1: [43746, 11591, 29644, 46194, 77080],
 2: [24484, 53612, 67830, 56866],
 3: [79712, 54691, 7590, 11591, 51942],
 4: [10274, 59170, 79525, 50695, 25129]}

In [65]:
instance_closure_all={
    ent: set([]).union(*[
        subclass_closure[cls]
        for cls in classes
    ])
    for ent, classes in tqdm(instance_no_closure.items())
}

100%|███████████████████████████████████████████████████████████████████████| 123182/123182 [00:02<00:00, 42405.10it/s]


In [66]:
{k: list(v)[:5] for k, v in list(subclass_closure.items())[:5]}

{'http://yago-knowledge.org/resource/wikicat_Austrian_socialists': ['http://yago-knowledge.org/resource/wordnet_object_100002684',
  'http://yago-knowledge.org/resource/wordnet_living_thing_100004258',
  'http://yago-knowledge.org/resource/wordnet_whole_100003553',
  'http://yago-knowledge.org/resource/wordnet_politician_110450303',
  'http://yago-knowledge.org/resource/wordnet_socialist_110618848'],
 'http://yago-knowledge.org/resource/wikicat_People_from_Haltern': ['http://yago-knowledge.org/resource/wordnet_object_100002684',
  'http://yago-knowledge.org/resource/wordnet_living_thing_100004258',
  'http://yago-knowledge.org/resource/wordnet_whole_100003553',
  'http://yago-knowledge.org/resource/wordnet_causal_agent_100007347',
  'http://yago-knowledge.org/resource/wikicat_People_from_Haltern'],
 'http://yago-knowledge.org/resource/wikicat_County_seats_in_Wisconsin': ['http://yago-knowledge.org/resource/wordnet_center_108523483',
  'http://yago-knowledge.org/resource/wordnet_object_

In [67]:
subclassof_all2id={
    subclass: set([]).union(*[
        [class2id[item] for item in subclass_closure[id2class[cls]]]
        for cls in classes
    ])
    for subclass, classes in tqdm(subclassof2id.items())
}

100%|████████████████████████████████████████████████████████████████████████| 93932/93932 [00:00<00:00, 187599.93it/s]


In [68]:
{k: list(v)[:5] for k, v in list(subclassof_all2id.items())[:5]}

{48108: [51587, 76424, 50347, 19758, 7854],
 64265: [43047, 11591, 18794, 62797, 32846],
 81913: [72449, 43746, 25315, 11591, 50347],
 78351: [43047, 11591, 18794, 62797, 32846],
 30543: [69920, 43047, 18794, 41227, 61514]}

In [69]:
{k: list(v)[:5] for k, v in list(instance_closure_all.items())[:5]}

{'http://yago-knowledge.org/resource/Şanver_Göymen': ['http://yago-knowledge.org/resource/wikicat_Turkish_people',
  'http://yago-knowledge.org/resource/wikicat_Altay_S.K._footballers',
  'http://yago-knowledge.org/resource/wordnet_football_player_110101634',
  'http://yago-knowledge.org/resource/yagoLegalActor',
  'http://yago-knowledge.org/resource/wordnet_hockey_player_110179291'],
 'http://yago-knowledge.org/resource/Qatar_national_under-23_football_team': ['http://yago-knowledge.org/resource/wordnet_abstraction_100002137',
  'http://yago-knowledge.org/resource/wordnet_social_group_107950920',
  'http://yago-knowledge.org/resource/wordnet_team_108208560',
  'http://yago-knowledge.org/resource/yagoLegalActor',
  'http://yago-knowledge.org/resource/wikicat_National_sports_teams_of_Qatar'],
 'http://yago-knowledge.org/resource/Main-Spessart': ['http://yago-knowledge.org/resource/wordnet_object_100002684',
  'http://yago-knowledge.org/resource/yagoPermanentlyLocatedEntity',
  'http://y

In [70]:
inst_type_all={
    ent2id[ent]: [class2id[cls] for cls in classes]
    for ent, classes in instance_closure_all.items()
}

In [71]:
{k: list(v)[:5] for k, v in list(inst_type_all.items())[:5]}

{0: [62822, 10637, 32485, 11591, 22179],
 1: [50347, 72449, 60644, 11591, 46194],
 2: [18794, 67830, 56866, 24484, 43047],
 3: [54691, 40569, 32485, 62635, 11591],
 4: [25129, 37876, 79525, 11591, 82930]}

In [72]:
class2ent={
    cls: [
        ent
        for ent, classes in instance_closure_all.items()
        if cls in classes
    ]
    for cls in tqdm(class2id.keys())
}

100%|████████████████████████████████████████████████████████████████████████████| 93937/93937 [36:59<00:00, 42.33it/s]


In [73]:
{k: list(v)[:5] for k, v in list(class2ent.items())[:5]}

{'http://yago-knowledge.org/resource/wikicat_Austrian_socialists': ['http://yago-knowledge.org/resource/Heinz_Fischer',
  'http://yago-knowledge.org/resource/Wilhelm_Reich',
  'http://yago-knowledge.org/resource/Martin_Buber',
  'http://yago-knowledge.org/resource/Franz_Kafka',
  'http://yago-knowledge.org/resource/Ernst_Mach'],
 'http://yago-knowledge.org/resource/wikicat_People_from_Haltern': ['http://yago-knowledge.org/resource/Christoph_Metzelder',
  'http://yago-knowledge.org/resource/Benedikt_Höwedes',
  'http://yago-knowledge.org/resource/Malte_Metzelder'],
 'http://yago-knowledge.org/resource/wikicat_County_seats_in_Wisconsin': ['http://yago-knowledge.org/resource/Oshkosh',
  'http://yago-knowledge.org/resource/Sparta,_Wisconsin',
  'http://yago-knowledge.org/resource/Kewaunee,_Wisconsin',
  'http://yago-knowledge.org/resource/Fond_du_Lac,_Wisconsin',
  'http://yago-knowledge.org/resource/Ashland,_Wisconsin'],
 'http://yago-knowledge.org/resource/wikicat_Military_operations_inv

In [74]:
classid2entid={
    class2id[cls]: [ent2id[ent] for ent in ent_list]
    for cls, ent_list in tqdm(class2ent.items())
}

100%|█████████████████████████████████████████████████████████████████████████| 93937/93937 [00:01<00:00, 83734.56it/s]


In [75]:
{k: list(v)[:5] for k, v in list(classid2entid.items())[:5]}

{0: [15334, 33040, 46963, 108905, 111715],
 1: [26721, 28601, 59716],
 2: [583, 2023, 13174, 13504, 14404],
 3: [5774, 120557],
 4: [20521, 43833, 44400, 46469, 68440]}

In [76]:
with open(f"{dest_pkl_folder}subclassof2id.pkl", "wb") as handle:
    dump(subclassof2id, handle)

with open(f"{dest_pkl_folder}subclassof_all2id.pkl", "wb") as handle:
    dump(subclassof_all2id, handle)

with open(f"{dest_pkl_folder}inst_type.pkl", "wb") as handle:
    dump(inst_type, handle)

with open(f"{dest_pkl_folder}inst_type_all.pkl", "wb") as handle:
    dump({k: set(v) for k, v in inst_type_all.items()}, handle)

with open(f"{dest_pkl_folder}classid2entid.pkl", "wb") as handle:
    dump(classid2entid, handle)